## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 8: Regularisierung (Dropout, BatchNorm)
# Niveau: Fortgeschrittene
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Winziger Datensatz (200 Samples) für gezieltes Overfitting ─────────────
(x_all, y_all), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_all  = x_all.astype("float32").reshape(-1, 784) / 255.0
x_test = x_test.astype("float32").reshape(-1, 784) / 255.0

# Nur 200 Trainingsbeispiele → leichtes Overfitting
x_klein = x_all[:200]
y_klein = y_all[:200]

# ── 2. Großes Modell (anfällig für Overfitting) ───────────────────────────────
def ueberparam_modell(
    mit_dropout=False,
    mit_l2=False,
    name="Basis"
):
    reg = tf.keras.regularizers.l2(0.001) if mit_l2 else None

    schichten = [
        tf.keras.layers.Dense(512, activation="relu",
                              kernel_regularizer=reg, input_shape=(784,)),
    ]
    if mit_dropout:
        schichten.append(tf.keras.layers.Dropout(0.5))

    schichten += [
        tf.keras.layers.Dense(256, activation="relu", kernel_regularizer=reg),
    ]
    if mit_dropout:
        schichten.append(tf.keras.layers.Dropout(0.4))

    schichten += [
        tf.keras.layers.Dense(128, activation="relu", kernel_regularizer=reg),
        tf.keras.layers.Dense(10,  activation="softmax"),
    ]

    m = tf.keras.Sequential(schichten, name=name)
    m.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return m

# ── 3. Schrittweise Verbesserung ──────────────────────────────────────────────
EPOCHEN = 60

konfigurationen = [
    {"mit_dropout": False, "mit_l2": False, "callbacks": [],
     "name": "1_Basis (Overfitting)"},
    {"mit_dropout": True,  "mit_l2": False, "callbacks": [],
     "name": "2_+ Dropout"},
    {"mit_dropout": True,  "mit_l2": True,  "callbacks": [],
     "name": "3_+ Dropout + L2"},
    {"mit_dropout": True,  "mit_l2": True,
     "callbacks": [tf.keras.callbacks.EarlyStopping(
         monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)],
     "name": "4_+ EarlyStopping"},
]

histories = []
ergebnisse = []

for k in konfigurationen:
    print(f"\nTrainiere: {k['name']}...")
    m = ueberparam_modell(
        mit_dropout=k["mit_dropout"],
        mit_l2=k["mit_l2"],
        name=k["name"]
    )
    hist = m.fit(
        x_klein, y_klein,
        epochs=EPOCHEN,
        batch_size=16,
        validation_data=(x_test, y_test),
        callbacks=k["callbacks"],
        verbose=0
    )
    test_acc = m.evaluate(x_test, y_test, verbose=0)[1]
    train_acc_final = hist.history["accuracy"][-1]
    val_acc_final   = hist.history["val_accuracy"][-1]
    overfitting_gap = train_acc_final - val_acc_final

    print(f"  Train-Acc: {train_acc_final:.4f} | Val-Acc: {val_acc_final:.4f} | "
          f"Gap: {overfitting_gap:.4f}")

    histories.append((k["name"], hist))
    ergebnisse.append({
        "Name":     k["name"],
        "Train":    train_acc_final,
        "Val":      val_acc_final,
        "Test":     test_acc,
        "Gap":      overfitting_gap,
        "Epochen":  len(hist.history["loss"]),
    })

# ── 4. Zusammenfassung ───────────────────────────────────────────────────────
print("\n── Zusammenfassung ──")
print(f"{'Konfiguration':<35} {'Train':>8} {'Val':>8} {'Test':>8} {'Gap':>8} {'Ep':>5}")
print("-" * 75)
for r in ergebnisse:
    print(f"{r['Name']:<35} {r['Train']:>8.4f} {r['Val']:>8.4f} "
          f"{r['Test']:>8.4f} {r['Gap']:>8.4f} {r['Epochen']:>5}")

# ── 5. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
farben = ["#e74c3c", "#f39c12", "#2ecc71", "#3498db"]

for i, (name, hist) in enumerate(histories):
    ep = range(1, len(hist.history["loss"]) + 1)
    axes[i].plot(ep, hist.history["accuracy"],     color=farben[i],
                 linestyle="-",  label="Training-Acc")
    axes[i].plot(ep, hist.history["val_accuracy"], color=farben[i],
                 linestyle="--", label="Val-Acc")
    gap = ergebnisse[i]["Gap"]
    axes[i].set_title(f"{name}\nOverfitting-Gap: {gap:.4f}")
    axes[i].set_xlabel("Epoche")
    axes[i].set_ylabel("Genauigkeit")
    axes[i].legend(fontsize=8)
    axes[i].grid(True)
    axes[i].set_ylim(0, 1.05)
    axes[i].fill_between(
        ep,
        hist.history["accuracy"],
        hist.history["val_accuracy"],
        alpha=0.15, color=farben[i], label="Overfitting-Bereich"
    )

plt.suptitle("Overfitting erkennen und schrittweise beheben\n"
             "(200 Trainingssamples, großes Modell)", fontsize=13)
plt.tight_layout()
plt.savefig("F8_1_overfitting_beheben.png", dpi=100)
plt.show()
print("Diagramm gespeichert: F8_1_overfitting_beheben.png")


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 8: Regularisierung (Dropout, BatchNorm)
# Niveau: Fortgeschrittene
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. MNIST-Daten ────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32").reshape(-1, 784) / 255.0
x_test  = x_test.astype("float32").reshape(-1, 784)  / 255.0

# ── 2. Modell MIT Dropout (der beim Inference aktiv bleibt) ───────────────────
modell = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation="relu", input_shape=(784,)),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation="softmax"),
], name="MC_Dropout_Modell")

modell.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ── 3. Training ───────────────────────────────────────────────────────────────
print("Trainiere Modell...")
modell.fit(x_train, y_train, epochs=10, batch_size=128,
           validation_split=0.1, verbose=1)

test_acc = modell.evaluate(x_test, y_test, verbose=0)[1]
print(f"Test-Genauigkeit (Standard): {test_acc:.4f}")

# ── 4. Monte Carlo Dropout: Unsicherheitsschätzung ───────────────────────────
print("\nBerechne MC-Dropout Unsicherheiten (T=100 Durchläufe)...")

N_STICHPROBEN = 100
N_BEISPIELE   = 500    # Teilmenge für Geschwindigkeit

x_probe = x_test[:N_BEISPIELE]
y_probe = y_test[:N_BEISPIELE]

# T stochastische Vorwärtsdurchläufe (training=True aktiviert Dropout)
stochastische_vorhersagen = np.zeros((N_STICHPROBEN, N_BEISPIELE, 10))
for t in range(N_STICHPROBEN):
    stochastische_vorhersagen[t] = modell(x_probe, training=True).numpy()

# Mittlere Vorhersage und Unsicherheit (Standardabweichung)
mittlere_vorhersage   = stochastische_vorhersagen.mean(axis=0)    # (N, 10)
unsicherheit          = stochastische_vorhersagen.std(axis=0)     # (N, 10)

# Vorhergesagte Klasse und maximale Unsicherheit
pred_klassen = np.argmax(mittlere_vorhersage, axis=1)
max_unsicher = unsicherheit[np.arange(N_BEISPIELE), pred_klassen]  # Unc. der pred. Klasse

# Korrekte / Falsche Vorhersagen
korrekt_mask = pred_klassen == y_probe

print(f"Genauigkeit (MC-Dropout):    {korrekt_mask.mean():.4f}")
print(f"Mittlere Unsicherheit (korrekt): {max_unsicher[korrekt_mask].mean():.4f}")
print(f"Mittlere Unsicherheit (falsch):  {max_unsicher[~korrekt_mask].mean():.4f}")

# ── 5. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# (a) Unsicherheitsverteilung: korrekt vs. falsch
axes[0].hist(max_unsicher[korrekt_mask],  bins=30, alpha=0.7,
             color="#2ecc71", label="Korrekte Vorhersagen", density=True)
axes[0].hist(max_unsicher[~korrekt_mask], bins=30, alpha=0.7,
             color="#e74c3c", label="Falsche Vorhersagen", density=True)
axes[0].set_title("MC-Dropout Unsicherheitsverteilung")
axes[0].set_xlabel("Unsicherheit (Std der Vorhersagen)")
axes[0].set_ylabel("Dichte")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# (b) Konfidenz vs. Unsicherheit
konfidenz = mittlere_vorhersage.max(axis=1)
axes[1].scatter(konfidenz[korrekt_mask],  max_unsicher[korrekt_mask],
                alpha=0.5, s=20, color="#2ecc71", label="Korrekt")
axes[1].scatter(konfidenz[~korrekt_mask], max_unsicher[~korrekt_mask],
                alpha=0.7, s=30, color="#e74c3c", label="Falsch", marker="x")
axes[1].set_title("Konfidenz vs. Unsicherheit")
axes[1].set_xlabel("Konfidenz (max. mittl. Vorhersage)")
axes[1].set_ylabel("Unsicherheit (Std)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# (c) Beispiele mit höchster Unsicherheit
sortiert_idx = np.argsort(max_unsicher)[::-1][:5]
for j, idx in enumerate(sortiert_idx):
    bild = x_probe[idx].reshape(28, 28)
    axes[2].imshow(bild, cmap="gray", extent=[j*30, j*30+25, 0, 28])
    axes[2].text(j*30+12, -3, f"P:{pred_klassen[idx]}\n"
                  f"U:{max_unsicher[idx]:.3f}",
                  ha="center", fontsize=7, color="#e74c3c" if not korrekt_mask[idx] else "#2ecc71")
axes[2].set_xlim(-2, 155)
axes[2].set_ylim(-8, 30)
axes[2].set_title("Top-5 unsicherste Vorhersagen\n(P=Pred, U=Unc)")
axes[2].axis("off")

plt.suptitle("Monte Carlo Dropout – Unsicherheitsschätzung\n"
             f"(T={N_STICHPROBEN} stochastische Durchläufe)", fontsize=13)
plt.tight_layout()
plt.savefig("F8_2_mc_dropout.png", dpi=100)
plt.show()
print("Diagramm gespeichert: F8_2_mc_dropout.png")


## Exercise 3

**Dataset Used:** CIFAR-10 (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 8: Regularisierung (Dropout, BatchNorm)
# Niveau: Fortgeschrittene
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. CIFAR-10 Teilmenge laden ───────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train = x_train[:10000].astype("float32") / 255.0
x_test  = x_test[:2000].astype("float32")   / 255.0
y_train = y_train[:10000].flatten()
y_test  = y_test[:2000].flatten()

# ── 2. Nicht-regularisiertes Baseline-Modell ──────────────────────────────────
def baseline_modell():
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu",
                               padding="same", input_shape=(32, 32, 3)),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(10,  activation="softmax"),
    ], name="Baseline_Ohne_Regularisierung")

# ── 3. Vollständig regularisiertes Modell ─────────────────────────────────────
def regularisiertes_modell():
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001,
        decay_steps=10000
    )
    modell = tf.keras.Sequential([
        # Block 1: BN + Dropout
        tf.keras.layers.Conv2D(
            64, (3, 3), padding="same", input_shape=(32, 32, 3),
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),

        # Block 2: BN + Dropout
        tf.keras.layers.Conv2D(
            128, (3, 3), padding="same",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),

        # Dense + BN + Dropout
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(
            512, kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10, activation="softmax"),
    ], name="Regularisiertes_Modell")
    return modell, lr_schedule

# ── 4. Callbacks ──────────────────────────────────────────────────────────────
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=4, verbose=1
)

# ── 5. Baseline trainieren ────────────────────────────────────────────────────
EPOCHEN = 30
print("Trainiere Baseline (ohne Regularisierung)...")
m_base = baseline_modell()
m_base.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])
h_base = m_base.fit(x_train, y_train, epochs=EPOCHEN, batch_size=64,
                     validation_data=(x_test, y_test), verbose=0)

# ── 6. Regularisiertes Modell trainieren ──────────────────────────────────────
print("Trainiere regularisiertes Modell...")
m_reg, _ = regularisiertes_modell()
m_reg.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
h_reg = m_reg.fit(
    x_train, y_train, epochs=EPOCHEN, batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=[early_stop, reduce_lr], verbose=0
)

# ── 7. Ergebnisse ─────────────────────────────────────────────────────────────
acc_base = m_base.evaluate(x_test, y_test, verbose=0)[1]
acc_reg  = m_reg.evaluate(x_test, y_test, verbose=0)[1]

print(f"\nTest-Genauigkeit Baseline:          {acc_base:.4f}")
print(f"Test-Genauigkeit Regularisiert:     {acc_reg:.4f}")
print(f"Verbesserung:                       {(acc_reg - acc_base)*100:+.2f}%")

# Overfitting-Gap
gap_base = h_base.history["accuracy"][-1] - h_base.history["val_accuracy"][-1]
gap_reg  = h_reg.history["accuracy"][-1]  - h_reg.history["val_accuracy"][-1]
print(f"\nOverfitting-Gap Baseline:           {gap_base:.4f}")
print(f"Overfitting-Gap Regularisiert:      {gap_reg:.4f}")

# ── 8. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(h_base.history["accuracy"],     color="#e74c3c", linestyle="-",
             label="Baseline (Train)")
axes[0].plot(h_base.history["val_accuracy"], color="#e74c3c", linestyle="--",
             label="Baseline (Val)")
axes[0].plot(h_reg.history["accuracy"],      color="#2ecc71", linestyle="-",
             label="Regularisiert (Train)")
axes[0].plot(h_reg.history["val_accuracy"],  color="#2ecc71", linestyle="--",
             label="Regularisiert (Val)")
axes[0].set_title("Genauigkeit: Baseline vs. Regularisiert")
axes[0].set_xlabel("Epoche")
axes[0].set_ylabel("Genauigkeit")
axes[0].legend(fontsize=8)
axes[0].grid(True)

axes[1].plot(h_base.history["val_loss"], color="#e74c3c",
             label="Baseline Val-Loss", linewidth=2)
axes[1].plot(h_reg.history["val_loss"],  color="#2ecc71",
             label="Regularisiert Val-Loss", linewidth=2)
axes[1].set_title("Validierungs-Verlust")
axes[1].set_xlabel("Epoche")
axes[1].set_ylabel("Verlust")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("Kombination: BatchNorm + Dropout + L2 + EarlyStopping + ReduceLR",
             fontsize=12)
plt.tight_layout()
plt.savefig("F8_3_kombinierte_regularisierung.png", dpi=100)
plt.show()
print("Diagramm gespeichert: F8_3_kombinierte_regularisierung.png")
